# 07 · Placebo — **HARD**

The same pipeline, the same directions, the same nulls — run against labels that
**cannot** carry information.

The corpus is `mix10_clean.jsonl`: 10,000 examples that are all source N. A
random 1,000 of them are labelled "A". There is no trait, no A completion, and
no difference of any kind between the two label groups, so every scorer must
come back at chance.

`L_student` is taken under the **clean** adapter, which is the honest
counterpart here: a "clean-of-clean" student would be trained on identical data
with an identical seed and therefore *is* the clean student.

**Gate:** every scorer's AUROC CI at the pre-registered layer must contain 0.5,
and no cell in the 29-layer grid may exceed its null p95. A failure means the
pipeline manufactures signal, and nothing in notebook 06 can be reported.

In [ ]:
# --- bootstrap: identical first cell in every pivot notebook -----------------
# /workspace is the Runpod network volume, so `runs/` (which config.py resolves
# relative to the repo root) survives a pod stop. Nothing here writes to the
# container disk except the HF cache, which is redirected for the same reason.
import os, sys, json, time, hashlib
from pathlib import Path

ROOT = Path("/workspace/subliminal-attrib")
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "src"))
os.environ.setdefault("SUBATTR_THIRD_PARTY", str(ROOT / "third_party"))
os.environ.setdefault("HF_HOME", "/workspace/hf_home")
os.environ.setdefault("WANDB_MODE", "disabled")

%load_ext autoreload
%autoreload 2

import torch
from subattr import config

cfg  = config.load("configs/pivot.yaml")
DATA = cfg.data_dir
RUN  = cfg.run_dir
MIX  = DATA / "mixtures"
T0   = time.time()

print(f"config    {cfg.name}   model_hash={cfg.hash}   data_hash={cfg.data_hash}")
print(f"git       {config.git_sha()}")
print(f"gpu       {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu'}")
print(f"data_dir  {DATA}")
print(f"run_dir   {RUN}")

In [ ]:
# The fraction chosen by the notebook-02 gate. Every stage after 02 reads it
# rather than hard-coding a dose, so the whole pipeline moves together if the
# gate is ever re-run.
GATE = json.loads((RUN / "gate.json").read_text())
FRACTION = GATE["fraction"]
print(f"gate fraction: {FRACTION}   (rule: {GATE['rule']})")

In [ ]:
from subattr import attribution as A
from subattr import baselines as bl
from subattr import ingest as ing
from subattr import metrics as M
from subattr import mixtures as mx
from subattr import train as tr
from subattr.cache import free_gpu, gpu_memory, load_tensors

import pandas as pd
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

deltas = load_tensors(RUN / "deltas.pt")
nulls = load_tensors(RUN / "nulls.pt")
LAYER = json.loads((RUN / "preregistered_layer.json").read_text())["layer"]
HEADLINE = ["delta_iso", "delta_mixed", "delta_clean", "delta_pureA"]
print(f"{len(deltas)} trait directions, {len(nulls)} null directions, layer {LAYER}")

In [ ]:
rows = ing.read_jsonl(MIX / "mix10_clean.jsonl")
N_PLANTED = 1000
fake_sources = mx.placebo_sources(len(rows), N_PLANTED, seed=cfg.seed)

RULE = f"{N_PLANTED} of {len(rows)} all-N examples labelled 'A' at random, then balanced"
subset = mx.balanced_subset(fake_sources, positive="A", seed=cfg.seed)
examples = [rows[i] for i in subset]
labels = [int(fake_sources[i] == "A") for i in subset]
print(f"placebo rule : {RULE}")
print(f"placebo set  : {len(examples)} examples, {sum(labels)} fake-A / "
      f"{len(labels) - sum(labels)} N -- all genuinely source N")

GRADCACHE = RUN / "gradcache_placebo"
LOSS_NAME = "loss_student_placebo.pt"
SCORES_NAME = "scores_placebo.parquet"
NULL_NAME = "null_placebo.parquet"
TABLE_NAME = "table_placebo.csv"
FIG_NAME = "fig_auroc_placebo.png"
STUDENT = "clean"
TITLE = "placebo (labels are noise)"

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(cfg.base_model)
base = AutoModelForCausalLM.from_pretrained(
    cfg.base_model, dtype=torch.bfloat16, device_map="auto"
).eval()
A.assert_no_adapter(base)   # gradients must be taken under the BASE model

A.cache_gradient_features(
    base, tokenizer, examples, GRADCACHE,
    chunk_size=250, token_grad_layer=LAYER, progress_every=100,
)
features = A.load_gradient_features(GRADCACHE)
print(f"cached {features['sum_response'].shape[0]} examples, "
      f"{features['sum_response'].shape[1]} layers")
print(f"[gpu] {gpu_memory()}")

In [ ]:
# The cache is only valid if mean_response really is sum / n_scored.
probe = A.score_from_cache(
    {k: v[:64] for k, v in features.items()}, {"delta_iso": deltas["delta_iso"]},
    aggregations=("sum_response", "mean_response"),
)
wide = probe.pivot_table(index=["example_index", "layer"], columns="aggregation", values="score")
n_scored = features["n_scored"][:64].float().numpy()
expected = n_scored.repeat(features["sum_response"].shape[1])
total = wide["sum_response"].to_numpy()
gap = abs(total - wide["mean_response"].to_numpy() * expected)
assert (gap / pd.Series(abs(total)).clip(lower=1e-6).to_numpy()).max() < 1e-3, (
    "mean_response is not sum_response / n_scored"
)
print("OK: mean_response == sum_response / n_scored")
print(f"scored tokens per example: median {int(pd.Series(n_scored).median())}, "
      f"range [{int(n_scored.min())}, {int(n_scored.max())}]")

In [ ]:
# L_base comes free from the gradient cache; only L_student needs a pass.
loss_base = features["loss"]
loss_path = RUN / LOSS_NAME
if loss_path.exists():
    loss_student = torch.load(loss_path, map_location="cpu", weights_only=True)
    print(f"[cache] student losses: loaded from {loss_path}")
else:
    student = PeftModel.from_pretrained(base, tr.latest_adapter(str(RUN / "students" / STUDENT)))
    student.eval()
    try:
        loss_student = bl.response_losses(student, tokenizer, examples)
    finally:
        base = student.unload()
    torch.save(loss_student, loss_path)
    print(f"[cache] student losses: saved to {loss_path}")

print(f"mean L_base {float(loss_base.mean()):.4f}   "
      f"mean L_student({STUDENT}) {float(loss_student.mean()):.4f}")
free_gpu(base, tokenizer)
print(f"[gpu] {gpu_memory()}")

In [ ]:
scores = A.score_from_cache(features, deltas)
scores = pd.concat([scores, bl.grad_norm_frame(features),
                    bl.loss_gap_frame(loss_base, loss_student)], ignore_index=True)
scores.to_parquet(RUN / SCORES_NAME, index=False)
print(f"{len(scores):,} rows -> {RUN / SCORES_NAME}")
print(sorted(scores.direction.unique()))

In [ ]:
# The null stays wide: melting 96 directions x 29 layers x 4 aggregations to
# long form is >100M rows, and every one of them is reduced to an AUROC anyway.
null_frames = []
names = list(nulls)
for start in range(0, len(names), 8):
    group = {k: nulls[k] for k in names[start : start + 8]}
    null_frames.append(M.auroc_grid(A.score_tensors(features, group), labels))
    print(f"  scored nulls {min(start + 8, len(names))}/{len(names)}", flush=True)
null = pd.concat(null_frames, ignore_index=True)
null.to_parquet(RUN / NULL_NAME, index=False)
print(f"{len(null):,} null cells -> {RUN / NULL_NAME}")

In [ ]:
table = M.scorer_table(
    scores, labels, k=int(sum(labels)), n_boot=1000, seed=cfg.seed,
    bootstrap_layers=[LAYER, -1], null=null,
)
table.to_csv(RUN / TABLE_NAME, index=False)

cols = ["direction", "aggregation", "auroc", "auroc_lo", "auroc_hi", "ap", "p_at_k",
        "null_random_p95", "null_random_pct", "null_random_p",
        "null_covrand_p95", "null_covrand_pct", "null_covrand_p"]
headline = table[table.layer.isin([LAYER, -1])].copy()
headline["order"] = headline.direction.map(
    {d: i for i, d in enumerate(HEADLINE + ["loss_gap", "grad_norm"])}
).fillna(99)
headline = headline.sort_values(["order", "aggregation"])
print(f"=== layer {LAYER} (pre-registered) + layer-free baselines ===")
print(headline[cols].to_string(index=False, float_format=lambda v: f"{v:7.4f}"))

In [ ]:
import matplotlib.pyplot as plt

grid = table[(table.layer >= 0) & (table.aggregation == "sum_response")]
pivot = grid.pivot(index="direction", columns="layer", values="auroc")
pivot = pivot.reindex([d for d in HEADLINE + ["grad_norm"] if d in pivot.index])

fig, ax = plt.subplots(figsize=(11, 2.6))
im = ax.imshow(pivot.to_numpy(), aspect="auto", cmap="RdBu_r", vmin=0.3, vmax=0.7)
ax.set_yticks(range(len(pivot.index)), pivot.index)
ax.set_xticks(range(0, pivot.shape[1], 2), pivot.columns[::2])
ax.set_xlabel("residual slot (0 = embedding)")
ax.axvline(LAYER, color="k", lw=1.2, ls="--")
ax.set_title(f"AUROC, sum_response, all layers -- {TITLE}  (dashed = pre-registered layer)")
fig.colorbar(im, ax=ax, shrink=0.8)
fig.tight_layout()
fig.savefig(RUN / FIG_NAME, dpi=140)
plt.show()

print("\nExploratory: this is a maximum over 29 correlated layers. Only the "
      "dashed column is a pre-registered number.")

## 7.1 · The gate

In [ ]:
at_layer = table[table.layer.isin([LAYER, -1])]
covers_chance = (at_layer.auroc_lo <= 0.5) & (at_layer.auroc_hi >= 0.5)
failed_ci = at_layer[~covers_chance]

def _p95(series):
    devs = sorted(abs(x - 0.5) for x in series)
    return devs[min(len(devs) - 1, int(0.95 * len(devs)))] + 0.5

grid = table[table.layer >= 0].merge(
    null.groupby(["aggregation", "layer"]).auroc.apply(_p95)
        .rename("null_p95_any").reset_index(),
    on=["aggregation", "layer"], how="left",
)
# `grad_norm` uses aggregation "none", which the nulls have no counterpart for,
# so its grid cells are covered by the CI check above and not by this one.
covered = grid.null_p95_any.notna()
exceeds = grid[covered & (abs(grid.auroc - 0.5) > (grid.null_p95_any - 0.5))]
print(f"grid cells with a matching null: {int(covered.sum())} of {len(grid)} "
      f"(the rest are aggregation='none' baselines)")

print(f"scorers whose CI at layer {LAYER} misses 0.5 : {len(failed_ci)}")
if len(failed_ci):
    print(failed_ci[["direction", "aggregation", "auroc", "auroc_lo", "auroc_hi"]].to_string(index=False))
print(f"grid cells above their null p95              : {len(exceeds)} of {len(grid)}")
if len(exceeds):
    print(exceeds.nlargest(10, "auroc")[["direction", "aggregation", "layer", "auroc", "null_p95_any"]].to_string(index=False))

assert len(failed_ci) == 0, (
    "PLACEBO FAILED: a scorer separates labels that carry no information. "
    "The pipeline manufactures signal; notebook 06's numbers are not reportable "
    "until this is understood."
)
assert len(exceeds) == 0, (
    "PLACEBO FAILED: a grid cell exceeds its empirical null on noise labels."
)
print("\nOK: every scorer is at chance on placebo labels. Notebook 06 is now reportable.")

In [ ]:
print(f"wall clock: {(time.time() - T0) / 60:.1f} min")

### Attended time

_Fill in before committing:_ **__ min** attended.
Copy the wall clock above and this figure into `docs/compute_log.md`.